### Phase III Work Report: Econometric Panel Assembly and Transaction Cost Mapping
    
The objective of this pipeline was to generate the definitive, relational datasets required for thesis analysis. This required the strict temporal alignment of microsecond level trades, the isolation of valid swap parameters, and the exact mapping of network transaction costs (gas) and macro block-space utilization.

### Methodology
1. **Dataset A (Primary Swaps):** The enriched swap files were consolidated and processed via an Asof backward time join, mapping each individual swap log to the exact 1-minute Binance WETH/USDT price immediately preceding the block's timestamp. Strict validation rules were enforced, dropping malformed transactions (e.g., zero-amount pokes) and ensuring each valid swap possessed exactly one positive (inflow) and one negative (outflow) vector.
2. **Dataset B (Gas Execution Costs):** I used a vectorized membership algorithm against a NumPy array of 64-bit target prefixes, the pipeline scanned millions of Ethereum blocks via HyperSync to isolate the exact gas expenditures and effective base fees paid by the initiators of the Dataset A swaps. A subsequent exact-match semi-join on the full 64-character hash removed reorganization duplicates and cryptographic false positives.
3. **Dataset C (Macro Block State):** A distinct query mapped the comprehensive state of the network for every block in the thesis timeframe, extracting total block gas limits, consumed gas, and base fees to calculate a holistic "gas utilization" metric representing overall network congestion.


In [9]:
from config import OUT
import polars as pl
from pathlib import Path
import os
from datetime import datetime
import time
import traceback
import numpy as np
import requests
import asyncio
import shutil
import glob
from hypersync import HypersyncClient, ClientConfig, Query, FieldSelection, BlockField, StreamConfig

print(f"Saving data to: {OUT}")

Saving data to: C:\Users\Pouyan\python\thesis\Proposal\FINAL\Thesis_Output


---

## Building Dataset A batches using 1-min prices

here we build the foundation of dataset A by joining our previously cleaned swap files with the one minute ethereum price data. it processes the files iteratively to prevent memory exhaustion. also formats the token amounts and calculates the exact US dollar values based on the matched timestamps using a backward lookup strategy.

In [19]:
# configuration and setup
OUT = Path("Dataset_A")
OUT.mkdir(exist_ok=True)

# canonical token addresses for pricing 
STABLES = ["0xa0b86991c6218b36c1d19d4a2e9eb0ce3606eb48",
           "0xdac17f958d2ee523a2206206994597c13d831ec7",
           "0x6b175474e89094c44da98b954eedeac495271d0f"]
WETH = "0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2"

print("Loading 1-minute ETH prices...")
prices = pl.read_parquet("binance_weth_1m_prices.parquet").sort("price_minute")

# Condition for a valid swap: exactly one positive flow, one negative flow
a0 = pl.col("amount0_f64")
a1 = pl.col("amount1_f64")
ok = ((a0 > 0) & (a1 < 0)) | ((a0 < 0) & (a1 > 0))

# ------------------------------------------------------------------------------------------------------
# iterative file processing
files = sorted(Path("swaps_usd").glob("*.parquet"))
print(f"Found {len(files)} batches. Building Dataset A...\n")

for f in files:
    # read and sort by time to prepare for the backward time join
    df = pl.read_parquet(f).sort("block_time")

    # execute the join to grab the exact minute level price just before the block
    df = df.join_asof(prices, left_on="block_time", right_on="price_minute", strategy="backward")

    # -------------------------------------------------------------------------------------------------
    # schema construction
    out = df.select([
        # --- Primary & Join Keys
        pl.col("transaction_hash").alias("tx_hash"), 
        pl.col("block_number"),
        pl.col("log_index"),                  # CRITICAL for uniqueness
        pl.col("block_time"),
        
        # --- Context
        pl.col("tx_from").fill_null(pl.col("sender")).alias("wallet"),
        pl.col("sender"),
        pl.col("recipient"),
        pl.lit("Uniswap V3").alias("project"),
        pl.col("pool_address"),
        
        # --- Tokens (Guarded by 'ok')
        pl.when(ok & (a0 > 0)).then(pl.col("token0"))
          .when(ok).then(pl.col("token1")).otherwise(None).alias("token_in"),
          
        pl.when(ok & (a0 < 0)).then(pl.col("token0"))
          .when(ok).then(pl.col("token1")).otherwise(None).alias("token_out"),

        # --- Adjusted (Human-readable) Amounts
        pl.when(ok & (a0 > 0)).then(pl.col("amount0_adj").abs())
          .when(ok).then(pl.col("amount1_adj").abs()).otherwise(None).alias("amount_in_float"),
          
        pl.when(ok & (a0 < 0)).then(pl.col("amount0_adj").abs())
          .when(ok).then(pl.col("amount1_adj").abs()).otherwise(None).alias("amount_out_float"),
          
        # --- Raw Amounts (Using the exact Int128 columns for flawless precision)
        pl.when(ok & (a0 > 0)).then(pl.col("amount0").abs())
          .when(ok).then(pl.col("amount1").abs()).otherwise(None).alias("amount_in_raw"),
          
        pl.when(ok & (a0 < 0)).then(pl.col("amount0").abs())
          .when(ok).then(pl.col("amount1").abs()).otherwise(None).alias("amount_out_raw"),

        # --- USD Pricing (using the minute-level eth_price_usd)
        pl.when(~ok).then(None)
          .when(pl.col("token0").is_in(STABLES)).then(pl.col("amount0_adj").abs())
          .when(pl.col("token1").is_in(STABLES)).then(pl.col("amount1_adj").abs())
          .when(pl.col("token0") == WETH).then(pl.col("amount0_adj").abs() * pl.col("eth_price_usd"))
          .when(pl.col("token1") == WETH).then(pl.col("amount1_adj").abs() * pl.col("eth_price_usd"))
          .otherwise(None).alias("amount_usd"),
          
        # --- Extra Metrics & Flags for downstream analysis
        pl.col("tick"),
        pl.col("sqrt_price_x96"),
        pl.col("liquidity"),         
        pl.col("eth_price_usd").alias("weth_price_at_block"),
        pl.col("amount_overflow"),
        (~ok).alias("malformed_legs")
    ])
    
    out.write_parquet(OUT / f.name, compression="zstd")
    print(f" {f.name}: {out.height:,} rows | Malformed: {out['malformed_legs'].sum():,} | Unpriced: {out['amount_usd'].is_null().sum():,}")

print("\n Dataset A is complete")


Loading 1-minute ETH prices...
Found 24 batches. Building Dataset A...

 uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_21500000_21700000.parquet: 2,586,295 rows | Malformed: 82 | Unpriced: 135,937
 uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_21700000_21900000.parquet: 3,255,821 rows | Malformed: 81 | Unpriced: 171,337
 uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_21900000_22100000.parquet: 3,438,146 rows | Malformed: 118 | Unpriced: 175,201
 uniswap_v3_swaps_2025-01-01_to_2025-04-01__blocks_22100000_22148000.parquet: 662,532 rows | Malformed: 26 | Unpriced: 34,279
 uniswap_v3_swaps_2025-04-01_to_2025-07-01__blocks_22148000_22348000.parquet: 3,245,002 rows | Malformed: 90 | Unpriced: 167,555
 uniswap_v3_swaps_2025-04-01_to_2025-07-01__blocks_22348000_22548000.parquet: 3,065,328 rows | Malformed: 95 | Unpriced: 161,619
 uniswap_v3_swaps_2025-04-01_to_2025-07-01__blocks_22548000_22748000.parquet: 2,496,671 rows | Malformed: 95 | Unpriced: 110,850
 uniswap_v3_swaps_2025-04-0

---

# Dataset A Health check 
this cell runs diagnostic checks on the newly built dataset parts to ensure structural integrity. it calculates unique primary keys and checks for missing pricing data to verify the data quality before finalizing the merge.


In [22]:
# Delete the old master file that is causing the schema conflict
old_file = "Dataset_A/dataset_a_master.parquet"
if os.path.exists(old_file):
    os.remove(old_file)
    print(f" Deleted legacy file causing schema conflict: {old_file}\n")

# Setup Polars printing
pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(20)

# count missing values for every column to catch bugs early
print("Running Dataset A Health Check...\n")
lf = pl.scan_parquet("Dataset_A/*.parquet")

# 3. Null counts for every column
print("first check for null counts per column")
print(lf.null_count().collect())

# verify logical rules like identical tokens or malformed legs
print("second check for structural integrity")
metrics = lf.select([
    pl.len().alias("Total Swaps (Rows)"),
    pl.struct("block_number", "log_index").n_unique().alias("Unique Primary Keys"),
    pl.col("tx_hash").n_unique().alias("Unique TX Hashes"),
    (pl.col("token_in") == pl.col("token_out")).sum().alias("Same Token (Should be 0)"),
    pl.col("malformed_legs").sum().alias("Malformed Legs"),
    pl.col("block_time").min().alias("Start Time"),
    pl.col("block_time").max().alias("End Time")
]).collect(engine="streaming")
print(metrics)

# locate any dates where pricing failed to merge properly
print("third check analyzing missing united states dollar values")
unpriced = lf.filter(pl.col("amount_usd").is_null())

date_check = unpriced.select([
    pl.col("block_time").dt.date().alias("date")
]).group_by("date").len().sort("date").head(5).collect()
print("Top dates with missing USD prices:")
print(date_check)


Running Dataset A Health Check...

first check for null counts per column
shape: (1, 22)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬───┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ tx_ ┆ blo ┆ log ┆ blo ┆ wal ┆ sen ┆ rec ┆ pro ┆ poo ┆ tok ┆ … ┆ amo ┆ amo ┆ amo ┆ amo ┆ tic ┆ sqr ┆ liq ┆ wet ┆ amo ┆ mal │
│ has ┆ ck_ ┆ _in ┆ ck_ ┆ let ┆ der ┆ ipi ┆ jec ┆ l_a ┆ en_ ┆   ┆ unt ┆ unt ┆ unt ┆ unt ┆ k   ┆ t_p ┆ uid ┆ h_p ┆ unt ┆ for │
│ h   ┆ num ┆ dex ┆ tim ┆ --- ┆ --- ┆ ent ┆ t   ┆ ddr ┆ in  ┆   ┆ _ou ┆ _in ┆ _ou ┆ _us ┆ --- ┆ ric ┆ ity ┆ ric ┆ _ov ┆ med │
│ --- ┆ ber ┆ --- ┆ e   ┆ u32 ┆ u32 ┆ --- ┆ --- ┆ ess ┆ --- ┆   ┆ t_f ┆ _ra ┆ t_r ┆ d   ┆ u32 ┆ e_x ┆ --- ┆ e_a ┆ erf ┆ _le │
│ u32 ┆ --- ┆ u32 ┆ --- ┆     ┆     ┆ u32 ┆ u32 ┆ --- ┆ u32 ┆   ┆ loa ┆ w   ┆ aw  ┆ --- ┆     ┆ 96  ┆ u32 ┆ t_b ┆ low ┆ gs  │
│     ┆ u32 ┆     ┆ u32 ┆     ┆     ┆     ┆     ┆ u32 ┆     ┆   ┆ t   ┆ --- ┆ --- ┆ u32 ┆     ┆ --- ┆     ┆ loc ┆ --- ┆ --- │
│     ┆     ┆     ┆     ┆    

---

## Finalizes Dataset A to our Thesis folder
this step finalizes dataset A by applying the exact temporal boundaries for the thesis proposal. it removes malformed transaction legs and exports the final pristine dataset. it also generates a specific list of unique transaction hashes to guide the subsequent gas data extraction.


In [25]:
# Define output directory 
output_dir = Path("./Thesis_Output")
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Define xact proposal timeframe
start_date = datetime(2025, 1, 1)
end_date = datetime(2026, 6, 30, 23, 59, 59)

print(f"Applying timeframe filters and exporting to {output_dir}...")
print("This may take a few minutes due to the large CSV writing process...\n")

# Create the execution plan (Filter timeframe and clean data)
lf = (
    pl.scan_parquet("Dataset_A/*.parquet")
    .filter(pl.col("block_time") >= start_date)
    .filter(pl.col("block_time") <= end_date)
    .filter(~pl.col("malformed_legs")) # Drop MEV pokes/zero amounts
    .drop_nulls(subset=["amount_usd", "token_in"]) # Drop unpriced altcoin pools
)

# Save as Parquet (Highly recommended for loading into future Python scripts)
parquet_path = os.path.join(output_dir, "Dataset_A_Final.parquet")
lf.sink_parquet(parquet_path, compression="zstd")
print(f" Saved Parquet successfully: {parquet_path}")

# 5. Save as CSV (For universal compatibility)
csv_path = os.path.join(output_dir, "Dataset_A_Final.csv")
lf.sink_csv(csv_path)
print(f" Saved CSV successfully: {csv_path}")

print("\n Export complete! our pristine Dataset A is ready for the thesis.")


Applying timeframe filters and exporting to Thesis_Output...
This may take a few minutes due to the large CSV writing process...

 Saved Parquet successfully: Thesis_Output\Dataset_A_Final.parquet
 Saved CSV successfully: Thesis_Output\Dataset_A_Final.csv

 Export complete! our pristine Dataset A is ready for the thesis.


In [28]:
print("Generating my_tx_hashes.csv for Gas Extraction...")
df = pl.scan_parquet("./Thesis_Output/Dataset_A_Final.parquet").select("tx_hash").unique().collect()
df.write_csv("my_tx_hashes.csv")
print(f"Saved {df.height} unique hashes.")

Generating my_tx_hashes.csv for Gas Extraction...
Saved 45532663 unique hashes.


---

## Hypersync Gas Data Puller

this is extremely time consuming, expect +10hr

In [46]:
"""
HyperSync gas scanner.
Streams every transaction in a block range and keeps only the ones
whose hash appears in HASH_FILE.

Key facts confirmed by diagnostic:
  - HyperSync has NO `receipts` entity.
  - gas_used / effective_gas_price live on the TRANSACTION object.
  - Values come back as hex strings ("0x0d442f").
  - Hashes come back WITH the 0x prefix.
  - `to_block` is EXCLUSIVE.
"""

# -----------------------------------------------------------------------------
# scanner configuration 

API_TOKEN     = "Hypersync Token" #put down your own token
HYPERSYNC_URL = "https://eth.hypersync.xyz/query"

START_BLOCK = 21_500_000          # proof-of-life start
END_BLOCK   = 25_431_200          # EXCLUSIVE upper bound

HASH_FILE  = "my_tx_hashes.csv"
OUTPUT_DIR = Path("./gas_data_hypersync")

FLUSH_ROW_LIMIT   = 50_000        # flush after this many matches
FLUSH_BLOCK_LIMIT = 50_000        # ...or this many blocks, whichever first

REQUEST_TIMEOUT = 120
MAX_RETRIES     = 8               # consecutive failures before giving up

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FILE = OUTPUT_DIR / "checkpoint.txt"
HASH_CACHE      = OUTPUT_DIR / "target_prefixes.npy"


# -------------------------------------------------------------------------------
# helper functions
def to_int(val):
    """HyperSync returns hex strings; tolerate ints and decimals too."""
    if val is None:
        return None
    if isinstance(val, int):
        return val
    if isinstance(val, str):
        v = val.strip()
        if not v:
            return None
        try:
            return int(v, 16) if v.startswith(("0x", "0X")) else int(v)
        except ValueError:
            return None
    return None


def load_targets() -> np.ndarray:
    """
    Load target hashes as a sorted uint64 array of their first 64 bits.
    8 bytes/hash -> ~382 MB for 47.7M hashes.
    """
    if HASH_CACHE.exists():
        arr = np.load(HASH_CACHE)
        print(f" Loaded {len(arr):,} hash prefixes from cache "
              f"({arr.nbytes / 1e6:.0f} MB).")
        return arr

    print("Loading hashes from CSV (first run — this is cached afterwards)...")
    df = pl.read_csv(HASH_FILE, columns=["tx_hash"])

    # Strip 0x, take first 16 hex chars (= 64 bits), parse as uint64.
    prefixes = (
        df.get_column("tx_hash")
          .cast(pl.Utf8)
          .str.to_lowercase()
          .str.strip_prefix("0x")
          .str.slice(0, 16)
          .drop_nulls()
          .to_list()
    )

    arr = np.fromiter(
        (int(p, 16) for p in prefixes if len(p) == 16),
        dtype=np.uint64,
        count=-1,
    )
    arr = np.unique(arr)          # sorts + dedupes in one shot
    np.save(HASH_CACHE, arr)

    print(f" Loaded {len(arr):,} unique hash prefixes "
          f"({arr.nbytes / 1e6:.0f} MB). Cached to {HASH_CACHE}.")
    return arr


def make_matcher(targets: np.ndarray):
    """Vectorized membership test against a sorted uint64 array."""
    n = len(targets)

    def match(values: np.ndarray) -> np.ndarray:
        if n == 0 or values.size == 0:
            return np.zeros(values.size, dtype=bool)
        idx = np.searchsorted(targets, values)
        np.clip(idx, 0, n - 1, out=idx)
        return targets[idx] == values

    return match


def get_start_block() -> int:
    if CHECKPOINT_FILE.exists():
        try:
            return int(CHECKPOINT_FILE.read_text().strip())
        except ValueError:
            pass
    return START_BLOCK


def build_payload(from_block: int) -> dict:
    return {
        "from_block": from_block,
        "to_block": END_BLOCK,            # exclusive
        "transactions": [{}],             # empty filter = every transaction
        "field_selection": {
            "transaction": [
                "block_number",
                "hash",
                "gas_used",
                "effective_gas_price",
            ]
        },
    }


# ----------------------------------------------------------------------------------
# Main streaming function
def main():
    targets = load_targets()
    is_match = make_matcher(targets)

    current_block      = get_start_block()
    last_flushed_block = current_block

    existing = sorted(OUTPUT_DIR.glob("gas_matched_*.parquet"))
    file_counter = (
        int(existing[-1].stem.split("_")[-1]) + 1 if existing else 1
    )

    buffer        = []
    total_matched = 0
    total_scanned = 0
    consecutive_failures = 0
    t0 = time.time()

    session = requests.Session()
    session.headers.update({
        "Content-Type": "application/json",
        "Authorization": f"Bearer {API_TOKEN}",
    })

    def flush(next_block: int):
        """Write buffer to disk, THEN advance the checkpoint."""
        nonlocal buffer, file_counter, last_flushed_block
        if buffer:
            path = OUTPUT_DIR / f"gas_matched_{file_counter:04d}.parquet"
            pl.DataFrame(
                buffer,
                schema={
                    "tx_hash": pl.Utf8,
                    "block_number": pl.UInt64,
                    "gas_used": pl.UInt64,
                    "effective_gas_price": pl.UInt64,
                },
            ).write_parquet(path, compression="zstd")
            print(f" Saved {path.name} ({len(buffer):,} rows)")
            buffer = []
            file_counter += 1
        CHECKPOINT_FILE.write_text(str(next_block))
        last_flushed_block = next_block

    print(f"\n Streaming blocks {current_block:,} → {END_BLOCK:,} (exclusive)\n")

    while current_block < END_BLOCK:
        try:
            resp = session.post(
                HYPERSYNC_URL,
                json=build_payload(current_block),
                timeout=REQUEST_TIMEOUT,
            )

            if resp.status_code != 200:
                consecutive_failures += 1
                wait = min(60, 2 ** consecutive_failures)
                print(f"API {resp.status_code}: {resp.text[:300]} "
                      f"— retry in {wait}s ({consecutive_failures}/{MAX_RETRIES})")
                if consecutive_failures >= MAX_RETRIES:
                    print(" Too many consecutive failures. Stopping.")
                    break
                time.sleep(wait)
                continue

            consecutive_failures = 0
            data = resp.json()

            # collect this response's transactions ------------------------------
            hashes, blocks, gas_used, gas_price = [], [], [], []
            for batch in data.get("data", []):
                if not isinstance(batch, dict):
                    continue
                for tx in batch.get("transactions", []):
                    h = tx.get("hash")
                    if not h:
                        continue
                    h = h.lower()
                    if h.startswith("0x"):
                        h = h[2:]
                    if len(h) != 64:
                        continue
                    hashes.append(h)
                    blocks.append(tx.get("block_number"))
                    gas_used.append(tx.get("gas_used"))
                    gas_price.append(tx.get("effective_gas_price"))

            total_scanned += len(hashes)

            # vectorized match --------------------------------------------------
            if hashes:
                prefixes = np.fromiter(
                    (int(h[:16], 16) for h in hashes),
                    dtype=np.uint64,
                    count=len(hashes),
                )
                hits = np.flatnonzero(is_match(prefixes))
                for i in hits:
                    buffer.append({
                        "tx_hash": "0x" + hashes[i],
                        "block_number": to_int(blocks[i]),
                        "gas_used": to_int(gas_used[i]),
                        "effective_gas_price": to_int(gas_price[i]),
                    })
                total_matched += len(hits)

            # advance ------------------------------------------------------------
            next_block = data.get("next_block")
            if not next_block or next_block <= current_block:
                print("  No forward progress from API — nudging by 1 block.")
                next_block = current_block + 1
            next_block = min(next_block, END_BLOCK)

            elapsed = time.time() - t0
            rate = total_scanned / elapsed if elapsed else 0
            done = next_block - START_BLOCK
            span = END_BLOCK - START_BLOCK
            print(f"→ block {next_block - 1:,} ({100 * done / span:5.2f}%) | "
                  f"scanned {total_scanned:,} | matched {total_matched:,} | "
                  f"{rate:,.0f} tx/s")

            if (len(buffer) >= FLUSH_ROW_LIMIT
                    or (next_block - last_flushed_block) >= FLUSH_BLOCK_LIMIT):
                flush(next_block)

            current_block = next_block

        except requests.exceptions.RequestException as e:
            consecutive_failures += 1
            wait = min(60, 2 ** consecutive_failures)
            print(f"Network issue: {e} — retry in {wait}s "
                  f"({consecutive_failures}/{MAX_RETRIES})")
            if consecutive_failures >= MAX_RETRIES:
                print(" Too many consecutive failures. Stopping.")
                break
            time.sleep(wait)

        except KeyboardInterrupt:
            print("\n⏸  Interrupted — flushing buffer before exit...")
            flush(current_block)
            print(f"Checkpoint saved at block {current_block:,}. "
                  f"Re-run to resume.")
            return

        except Exception as e:
            print(f"\n Script error: {e}")
            traceback.print_exc()
            time.sleep(5)

    flush(current_block)
    elapsed = time.time() - t0
    print(f"\n DONE in {elapsed / 3600:.2f}h — "
          f"scanned {total_scanned:,} txs, matched {total_matched:,}.")
    print(" Now run reconcile.py to drop any prefix false-positives.")

if __name__ == "__main__":
    main()


 Loaded 45,532,663 hash prefixes from cache (364 MB).

 Streaming blocks 25,431,200 → 25,431,200 (exclusive)


 DONE in 0.00h — scanned 0 txs, matched 0.
 Now run reconcile.py to drop any prefix false-positives.


---

## Gas Match Reconciliation
because the gas scanner in the previous cell matched only the first sixteen characters of the hashes to save memory, this cell performs a rigorous exact join against the full sixty four character string. this removes any false positive matches and calculates the final fee paid in wei.

In [49]:
# exact join reconciliation
OUTPUT_DIR = Path("./gas_data_hypersync")
HASH_FILE  = "my_tx_hashes.csv"
FINAL_OUT  = OUTPUT_DIR / "gas_final.parquet"

matched = pl.scan_parquet(OUTPUT_DIR / "gas_matched_*.parquet")

targets = (
    pl.scan_csv(HASH_FILE)
      .select(
          pl.col("tx_hash").cast(pl.Utf8).str.to_lowercase()
            .str.strip_prefix("0x").alias("_key")
      )
      .unique()
)

# execute an exact match join to drop reorg duplicates and prefix collisions
final = (
    matched
    .with_columns(
        pl.col("tx_hash").str.to_lowercase().str.strip_prefix("0x").alias("_key")
    )
    .join(targets, on="_key", how="semi")     # exact 64-char match
    .unique(subset=["_key"])                  # de-dupe reorg duplicates
    .drop("_key")
    .with_columns(
        (pl.col("gas_used") * pl.col("effective_gas_price")).alias("fee_wei")
    )
    .collect(streaming=True)
)

final.write_parquet(FINAL_OUT, compression="zstd")

raw_n = pl.scan_parquet(OUTPUT_DIR / "gas_matched_*.parquet").select(pl.len()).collect().item()
tgt_n = targets.select(pl.len()).collect().item()

print(f"Raw matched rows : {raw_n:,}")
print(f"After exact join : {len(final):,}")
print(f"Dropped          : {raw_n - len(final):,}")
print(f"Target hashes    : {tgt_n:,}")
print(f"Coverage         : {100 * len(final) / tgt_n:.4f}%")
print(f"\n Wrote {FINAL_OUT}")
print(final.head())


C:\Users\Pouyan\AppData\Local\Temp\ipykernel_15036\2902437881.py:29: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True)


Raw matched rows : 45,532,663
After exact join : 45,532,663
Dropped          : 0
Target hashes    : 45,532,663
Coverage         : 100.0000%

 Wrote gas_data_hypersync\gas_final.parquet
shape: (5, 5)
┌───────────────────────────────┬──────────────┬──────────┬─────────────────────┬──────────────────┐
│ tx_hash                       ┆ block_number ┆ gas_used ┆ effective_gas_price ┆ fee_wei          │
│ ---                           ┆ ---          ┆ ---      ┆ ---                 ┆ ---              │
│ str                           ┆ u64          ┆ u64      ┆ u64                 ┆ u64              │
╞═══════════════════════════════╪══════════════╪══════════╪═════════════════════╪══════════════════╡
│ 0xa19911c278e5754b2f9413075c9 ┆ 21623490     ┆ 261518   ┆ 13160124952         ┆ 3441609557197136 │
│ 0…                            ┆              ┆          ┆                     ┆                  │
│ 0x7e9549bef092efa7582a1061f07 ┆ 21623542     ┆ 420125   ┆ 10581830868         ┆ 444569169341

---

## Dataset B final alignment

This cell guarantees perfect relational integrity between our primary swap data (Dataset A) and transaction cost data (Dataset B). By applying a semi join using the unique transaction hashes from Dataset A, it filters out any extraneous gas records, ensuring that Dataset B strictly contains the exact network fee metrics for the finalized swap sample

In [52]:
# Define directory
output_dir = Path("./Thesis_Output")

# dataset alignment
print("1. Loading finalized Dataset A to get the exact transactions we need...")
# We only need the unique tx_hashes from the A dataset we just created
valid_txs = (
    pl.scan_parquet(os.path.join(output_dir, "Dataset_A_Final.parquet"))
    .select("tx_hash")
    .unique()
)

print("2. Filtering Dataset B (Gas) to match Dataset A exactly...")
# Load raw gas data and keep ONLY the rows whose tx_hash exists in valid_txs
df_b = (
    pl.scan_parquet("gas_data_hypersync/gas_final.parquet")
    .join(valid_txs, on="tx_hash", how="semi")
)

# Save as Parquet
parquet_path = os.path.join(output_dir, "Dataset_B_Final.parquet")
df_b.sink_parquet(parquet_path, compression="zstd")
print(f" Saved Parquet successfully: {parquet_path}")

# Save as CSV
csv_path = os.path.join(output_dir, "Dataset_B_Final.csv")
df_b.sink_csv(csv_path)
print(f" Saved CSV successfully: {csv_path}")

print("\n Dataset B is now aligned with Dataset A and safely stored")


1. Loading finalized Dataset A to get the exact transactions we need...
2. Filtering Dataset B (Gas) to match Dataset A exactly...
 Saved Parquet successfully: Thesis_Output\Dataset_B_Final.parquet
 Saved CSV successfully: Thesis_Output\Dataset_B_Final.csv

 Dataset B is now perfectly aligned with Dataset A and safely stored


---

## Dataset C (Macro Block Metrics) Extraction
This cell constructs Dataset C, which captures the macro-level state of the Ethereum network for every single block in our timeframe. It configures the HyperSync client to force the inclusion of all blocks (bypassing log filters), downloads the raw base fees and gas limits, and parses the hexadecimal byte strings into clean integer metrics representing network congestion.

In [58]:
# 1. Config
API_TOKEN = "HyperSync Token" #put down your own RPC
START_BLOCK = 21_500_000
END_BLOCK = 25_431_200
RAW_DIR = "dataset_c_raw"
OUTPUT_DIR = Path("./Thesis_Output")

async def fetch_blocks():
    client = HypersyncClient(ClientConfig(
        url="https://eth.hypersync.xyz",
        bearer_token=API_TOKEN,
    ))
    
    q = Query(
        from_block=START_BLOCK,
        to_block=END_BLOCK,
        include_all_blocks=True,   # <-- CRITICAL FIX: forces ALL blocks to be returned,
                                    #     not just ones tied to matching logs/transactions
        field_selection=FieldSelection(
            block=[
                BlockField.NUMBER,
                BlockField.TIMESTAMP,
                BlockField.BASE_FEE_PER_GAS,
                BlockField.GAS_LIMIT,
                BlockField.GAS_USED
            ]
        )
    )
    
    print(f"Fetching blocks {START_BLOCK:,} to {END_BLOCK:,}...")
    if os.path.exists(RAW_DIR):
        shutil.rmtree(RAW_DIR)
        
    config = StreamConfig()
    await client.collect_parquet(RAW_DIR, q, config)
    
    print(" Download complete!")

def debug_show_tree(base_dir):
    print(f"\n Contents of '{base_dir}':")
    if not os.path.exists(base_dir):
        print("    Directory does not exist!")
        return
    for root, dirs, files in os.walk(base_dir):
        level = root.replace(base_dir, "").count(os.sep)
        indent = "   " * level
        print(f"{indent}{os.path.basename(root) or base_dir}/")
        for f in files:
            print(f"{indent}   {f}")

def find_blocks_parquet(base_dir):
    direct = os.path.join(base_dir, "blocks.parquet")
    if os.path.isfile(direct):
        return direct
    
    blocks_dir = os.path.join(base_dir, "blocks")
    if os.path.isdir(blocks_dir):
        pattern = os.path.join(blocks_dir, "*.parquet")
        if glob.glob(pattern):
            return pattern
    
    matches = glob.glob(os.path.join(base_dir, "**", "*.parquet"), recursive=True)
    block_matches = [m for m in matches if "block" in m.lower()]
    
    if block_matches:
        return block_matches if len(block_matches) > 1 else block_matches[0]
    if matches:
        return matches
    
    raise FileNotFoundError(
        f"No parquet files found anywhere under '{base_dir}'. "
        f"The download may have failed silently."
    )

def build_dataset_c():
    print("\nProcessing Dataset C...")
    debug_show_tree(RAW_DIR)
    
    blocks_path = find_blocks_parquet(RAW_DIR)
    print(f"\n Found blocks data at: {blocks_path}")
    
    lf = pl.scan_parquet(blocks_path)
    schema = lf.schema
    
    def to_uint(col_name):
        dtype = schema[col_name]
        
        if dtype == pl.Utf8:
            # Hex string like "0x1a2b" -> strip prefix, parse base 16
            return (
                pl.col(col_name)
                .str.strip_prefix("0x")
                .str.to_integer(base=16, strict=False)
            )
        elif dtype == pl.Binary:
            # Raw bytes -> encode to hex string first, THEN parse as integer
            return (
                pl.col(col_name)
                .bin.encode("hex")
                .str.to_integer(base=16, strict=False)
            )
        elif dtype in (pl.Int8, pl.Int16, pl.Int32, pl.Int64,
                       pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64):
            # Already numeric — no conversion needed
            return pl.col(col_name)
        else:
            raise TypeError(f"Unexpected dtype '{dtype}' for column '{col_name}'")
    
    dataset_c = (
        lf.select([
            to_uint("number").cast(pl.UInt64).alias("block_number"),
            to_uint("timestamp").cast(pl.Int64).alias("ts_raw"),
            to_uint("base_fee_per_gas").cast(pl.UInt64).alias("base_fee"),
            to_uint("gas_limit").cast(pl.UInt64).alias("gas_limit"),
            to_uint("gas_used").cast(pl.UInt64).alias("gas_used"),
        ])
        .with_columns([
            pl.from_epoch(pl.col("ts_raw"), time_unit="s").alias("timestamp"),
            (pl.col("gas_used") / pl.col("gas_limit")).alias("gas_utilization")
        ])
        .drop("ts_raw")
        .sort("block_number")
    ).collect()

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    
    parquet_path = os.path.join(OUTPUT_DIR, "Dataset_C_Final.parquet")
    dataset_c.write_parquet(parquet_path, compression="zstd")
    print(f" Saved Parquet successfully: {parquet_path}")
    
    csv_path = os.path.join(OUTPUT_DIR, "Dataset_C_Final.csv")
    dataset_c.write_csv(csv_path)
    print(f" Saved CSV successfully: {csv_path}")
    
    print("\nDataset C Preview:")
    print(dataset_c.head())
    print(f"\nTotal rows: {dataset_c.height:,}")

await fetch_blocks()
build_dataset_c()


Fetching blocks 21,500,000 to 25,431,200...
 Download complete!

Processing Dataset C...

 Contents of 'dataset_c_raw':
dataset_c_raw/
   blocks.parquet

 Found blocks data at: dataset_c_raw\blocks.parquet


C:\Users\Pouyan\AppData\Local\Temp\ipykernel_15036\2500653674.py:83: PerformanceWarning: Resolving the schema of a LazyFrame is a potentially expensive operation. Use `LazyFrame.collect_schema()` to get the schema without this warning.
  schema = lf.schema


 Saved Parquet successfully: Thesis_Output\Dataset_C_Final.parquet
 Saved CSV successfully: Thesis_Output\Dataset_C_Final.csv

Dataset C Preview:
shape: (5, 6)
┌──────────────┬────────────┬───────────┬──────────┬─────────────────────┬─────────────────┐
│ block_number ┆ base_fee   ┆ gas_limit ┆ gas_used ┆ timestamp           ┆ gas_utilization │
│ ---          ┆ ---        ┆ ---       ┆ ---      ┆ ---                 ┆ ---             │
│ u64          ┆ u64        ┆ u64       ┆ u64      ┆ datetime[μs]        ┆ f64             │
╞══════════════╪════════════╪═══════════╪══════════╪═════════════════════╪═════════════════╡
│ 21500000     ┆ 3790816777 ┆ 30000000  ┆ 13776378 ┆ 2024-12-28 09:14:47 ┆ 0.4592126       │
│ 21500001     ┆ 3752162387 ┆ 30000000  ┆ 13645926 ┆ 2024-12-28 09:14:59 ┆ 0.4548642       │
│ 21500002     ┆ 3709823175 ┆ 30000000  ┆ 14440144 ┆ 2024-12-28 09:15:11 ┆ 0.481338        │
│ 21500003     ┆ 3692515119 ┆ 30000000  ┆ 12023301 ┆ 2024-12-28 09:15:23 ┆ 0.4007767       │
│ 2

---

### Results & Data Integrity
architecture outputs three distinctly normalized datasets (A, B, and C) stored in compressed Parquet and CSV formats. Extensive diagnostic matrices confirmed zero incidence of schema conflicts, verifiable primary key uniqueness and perfect relational integrity across the distinct data domains. 